Validação dos modelos sazonais

In [ ]:
from pathlib import Path
import sys

import joblib
import pandas as pd

sys.path.append("/code/scripts")

from seasonal_utils import classification_metrics
from validation import pseudo_roc_blocks
from glass.wt import obj_to_tbl

In [ ]:
# ALTERAR APENAS ESTA VARIÁVEL

area = "centro"
year = 2025


samples_file = Path(
    f"/code/data/processed/{area}/model_inputs/seasonal/"
    "samples_valid_2025.csv"
)


# Áreas ardidas entre 15 de junho e 15 de setembro.
raster_reference = Path(
    f"/code/data/processed/{area}/area_ardida/effis/"
    f"raster_target_yearly/rst_ba_{year}_target.tif"
)


out_dir = Path(
    f"/code/data/results/{area}/seasonal_validation"
)

out_dir.mkdir(
    parents=True,
    exist_ok=True
)


print("Área:", area)
print("Ano:", year)
print("Amostra de validação:", samples_file)
print("Referência sazonal:", raster_reference)
print("Output:", out_dir)

## A. Validação convencional sobre os pontos de 2025

In [ ]:
valid = pd.read_csv(
    samples_file
)


point_model_defs = {
    "lr_susc": {
        "scenario": "C7",
        "model": "susc",
        "features": ["susc_lr"]
    },
    "rf_susc": {
        "scenario": "C8",
        "model": "susc",
        "features": ["susc_rf"]
    },
    "ssr_only": {
        "scenario": "C7",
        "model": "ssr_abs",
        "features": ["ssr_abs"]
    },
    "lr_susc_ssr": {
        "scenario": "C7",
        "model": "combined",
        "features": ["susc_lr", "ssr_abs"]
    },
    "rf_susc_ssr": {
        "scenario": "C8",
        "model": "combined",
        "features": ["susc_rf", "ssr_abs"]
    }
}


point_rows = []
prediction_rows = []


for validation_model, cfg in point_model_defs.items():
    scenario = cfg["scenario"]
    model_name = cfg["model"]
    features = cfg["features"]

    model_file = Path(
        f"/code/data/results/{area}/{scenario}/"
        f"seasonal/models/logit_{model_name}.joblib"
    )

    model = joblib.load(
        model_file
    )

    x = valid[features].to_numpy(
        dtype="float64"
    )

    probability = model.predict_proba(
        x
    )[:, 1]

    predicted = (
        probability >= 0.5
    ).astype("int16")

    metrics = classification_metrics(
        observed=valid["burned"],
        probability=probability,
        threshold=0.5
    )

    point_rows.append({
        "area": area,
        "validation_model": validation_model,
        "scenario": scenario,
        "source_model": model_name,
        "features": ", ".join(features),
        **metrics
    })

    part = valid[
        ["year", "row", "col", "x", "y", "burned"]
    ].copy()

    part["validation_model"] = validation_model
    part["scenario"] = scenario
    part["probability"] = probability
    part["predicted"] = predicted

    prediction_rows.append(
        part
    )


point_metrics = pd.DataFrame(
    point_rows
)

point_predictions = pd.concat(
    prediction_rows,
    ignore_index=True
)

point_metrics

## B. Validação raster comparável com C1–C6

In [ ]:
raster_model_defs = {
    "lr_susc": {
        "scenario": "C7",
        "model": "susc"
    },
    "rf_susc": {
        "scenario": "C8",
        "model": "susc"
    },
    "ssr_only": {
        "scenario": "C7",
        "model": "ssr_abs"
    },
    "lr_susc_ssr": {
        "scenario": "C7",
        "model": "combined"
    },
    "rf_susc_ssr": {
        "scenario": "C8",
        "model": "combined"
    }
}


raster_rows = []
curves = {}


for validation_model, cfg in raster_model_defs.items():
    scenario = cfg["scenario"]
    model_name = cfg["model"]

    raster = Path(
        f"/code/data/results/{area}/{scenario}/"
        f"seasonal/maps/prob_{model_name}_{year}.tif"
    )

    curve, auc_value, _ = pseudo_roc_blocks(
        ref=str(raster_reference),
        perigo_rst=str(raster),
        posval=1,
        otbl=None,
        block_size=1024
    )

    curve_name = (
        f"{validation_model}_{year}"
    )

    curve_file = out_dir / (
        f"{curve_name}_curve.xlsx"
    )

    obj_to_tbl(
        curve,
        str(curve_file)
    )

    curves[curve_name] = curve

    raster_rows.append({
        "area": area,
        "validation_model": validation_model,
        "scenario": scenario,
        "source_model": model_name,
        "year": year,
        "reference_period": "15 June–15 September",
        "auc": auc_value,
        "raster": str(raster),
        "curve_file": str(curve_file)
    })

    print(
        validation_model,
        round(auc_value, 6)
    )


raster_metrics = pd.DataFrame(
    raster_rows
)

raster_metrics

## Guardar os resultados

In [ ]:
results = out_dir / f"seasonal_validation_{year}.xlsx"

with pd.ExcelWriter(results) as writer:
    point_metrics.to_excel(
        writer,
        sheet_name="point_metrics",
        index=False
    )

    point_predictions.to_excel(
        writer,
        sheet_name="point_predictions",
        index=False
    )

    raster_metrics.to_excel(
        writer,
        sheet_name="raster_metrics",
        index=False
    )

print("Resultados guardados em:", results)